# Modeling the Minnesota Timberwolves with the Ornstein–Uhlenbeck Process

Stochastic-processes class project notebook.

**Goal.** Treat game-by-game point differential as a mean-reverting OU process,
fit parameters $(\mu, \theta, \sigma)$ to the 2023-24 and 2024-25 Timberwolves
seasons, and run 10,000 Monte Carlo simulations to project a full 82-game season.

**Model.**
$$
dX_t = \theta(\mu - X_t)\,dt + \sigma\,dW_t
$$

Discrete one-game transition (exact):
$$
X_{t+1}\mid X_t \sim \mathcal N\!\big(X_t e^{-\theta} + \mu(1-e^{-\theta}),\;
   \sigma^2\,(1-e^{-2\theta})/(2\theta)\big).
$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_loader import load_games
from ou_model import fit_both, fit_ols, fit_mle, stationary_std
from simulate import simulate_paths
from win_probability import (wins_from_paths, stationary_win_probability,
                              per_step_win_probability)

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True

## 1. Data

Pull 2023-24 and 2024-25 regular-season game logs from `nba_api`.
`POINT_DIFF` is signed (MIN points − opponent points).

In [ ]:
df = load_games(use_cache=True)
df.groupby('SEASON').agg(games=('WIN','count'), wins=('WIN','sum'),
                          avg_diff=('POINT_DIFF','mean'),
                          std_diff=('POINT_DIFF','std'))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for season, c in [('2023-24', '#236192'), ('2024-25', '#78BE20')]:
    sub = df[df.SEASON == season]
    ax.plot(sub.GAME_DATE, sub.POINT_DIFF, 'o-', ms=3.5, lw=1, alpha=.85,
            color=c, label=season)
ax.axhline(0, color='black', lw=.6, ls='--', alpha=.5)
ax.set(xlabel='Game date', ylabel='Point differential', title='MIN game-by-game point differential')
ax.legend();

## 2. Fit OU parameters

We use both:

* **OLS** on the AR(1) form $X_{t+1}=a+bX_t+e$ (closed form), and
* **MLE** maximizing the exact discrete-time conditional likelihood.

In [ ]:
x = df['POINT_DIFF'].to_numpy(dtype=float)
fits = fit_both(x)
for name, p in fits.items():
    print(f'{name}: mu={p.mu:.3f}  theta={p.theta:.4f}  sigma={p.sigma:.3f}  '
          f'stat_sd={stationary_std(p):.3f}  loglik={p.loglik:.2f}')
best = fits['MLE']

**Identifiability note.** When game-to-game autocorrelation is small (as it is
for NBA point differentials), $b=e^{-\theta}$ is near 0 and $\theta$ becomes
weakly identified individually, but the stationary distribution
$\mathcal N(\mu,\,\sigma^2/(2\theta))$ is well identified. Both estimators
agree on $\mu$ and on the stationary SD — those are what drive predictions.

In [ ]:
rows = []
for season in ['2023-24', '2024-25']:
    xs = df.loc[df.SEASON == season, 'POINT_DIFF'].to_numpy(dtype=float)
    for n, p in fit_both(xs).items():
        rows.append([season, n, p.mu, p.theta, p.sigma, stationary_std(p)])
pd.DataFrame(rows, columns=['season','method','mu','theta','sigma','stat_sd']).round(3)

## 3. Monte Carlo: 10,000 simulated 82-game seasons

In [ ]:
x0 = float(df['POINT_DIFF'].iloc[-1])
paths = simulate_paths(best, x0=x0, n_steps=82, n_paths=10_000, seed=42)
paths.shape

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for i in range(60):
    ax.plot(paths[i], lw=.6, alpha=.35, color='#236192')
ax.plot(paths.mean(0), color='crimson', lw=2, label='mean path')
ax.fill_between(np.arange(82), np.quantile(paths,.05,0), np.quantile(paths,.95,0),
                color='crimson', alpha=.15, label='5–95% band')
ax.axhline(best.mu, color='black', ls='--', lw=.8, label=f'μ={best.mu:.2f}')
ax.set(xlabel='Game', ylabel='Point differential',
       title=f'10,000 simulated paths (60 shown), starting x0={x0:.1f}')
ax.legend();

In [ ]:
wins = wins_from_paths(paths)
p_inf = stationary_win_probability(best)
print(f'Stationary P(win)  = {p_inf:.3f}')
print(f'Stationary E[wins] = {p_inf*82:.1f}')
print(f'Monte-Carlo mean wins = {wins.mean():.2f}')
print(f'Monte-Carlo 5–95% wins = [{np.percentile(wins,5):.0f}, {np.percentile(wins,95):.0f}]')

fig, ax = plt.subplots(figsize=(10, 4))
bins = np.arange(wins.min(), wins.max()+2) - .5
ax.hist(wins, bins=bins, color='#78BE20', edgecolor='white', alpha=.85)
ax.axvline(wins.mean(), color='crimson', lw=2, label=f'mean={wins.mean():.1f}')
ax.set(xlabel='Total wins', ylabel='# of simulations',
       title='Projected total wins for an 82-game season')
ax.legend();

## 4. Closed-form per-game win probability

In [ ]:
p_t = per_step_win_probability(best, x0=x0, n_steps=82)
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(np.arange(1,83), p_t, color='#236192', lw=2)
ax.axhline(p_inf, color='crimson', ls='--', label=f'stationary P(win)={p_inf:.3f}')
ax.set(xlabel='Game t', ylabel='P(point diff > 0)', ylim=(0,1),
       title=f'Conditional P(win) at game t (start x0={x0:.1f})')
ax.legend();

## 5. Takeaways for the report

* **True talent** $\mu \approx 5.8$ points: Minnesota is a clearly above-average team.
* **Volatility** is large (stationary SD ≈ 13 points), so single-game swings of
  ±20 are normal — fans over-reacting to bad nights are *literally* fitting noise.
* **Reversion** is fast: the AR(1) coefficient $b = e^{-\theta}$ is near 0, meaning
  there is essentially no game-to-game momentum. Streaks are statistical illusions.
* **Predicted record** for an 82-game season under the fitted model: roughly
  **55 wins** (5–95% interval ~[48, 62]).
* The per-season fits show a small drop from $\mu = 6.57$ (2023-24) to
  $\mu = 5.17$ (2024-25), a real but modest decline in true talent.